# 01 · EDA, target definition and baseline model

**Owner:** Machine learning · **Phase:** 1 → 2

What this notebook does, in order:

1. Lists what's in the GCS landing bucket and loads the latest snapshot of each source
2. **Recency check**: is the "legacy" inspections dataset still getting new records?
3. Profiles the inspection columns (result codes, violation levels, license categories)
4. Reshapes **violation rows → one row per inspection**
5. Defines the **target**: `1` if a routine inspection records any critical (`***`) violation
6. Builds leakage-safe features (only data from *before* each inspection)
7. Checks the licenses and 311 feeds (join keys, complaint types, geocoding)
8. Time-based split and two baselines: logistic regression and gradient boosting

**Before running:** `gcloud auth application-default login` with an account that can read the bucket.
To run on local copies instead, set `LOCAL_DATA_DIR` (same folder layout as the bucket).

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 160)

## 0. Config

Edit these if the bucket layout differs. Section 1 prints what's actually there.

In [ ]:
PROJECT_ID = os.getenv("GCP_PROJECT_ID", "ba882-team10-inspections")
BUCKET = os.getenv("GCS_LANDING_BUCKET", "ba882-team10-inspections-landing")
RAW_PREFIX = "raw"

# Set to a local folder (e.g. "../../data/raw") to skip GCS
LOCAL_DATA_DIR = os.getenv("LOCAL_DATA_DIR")

# Folder names under raw/ for each source. Fix after looking at the inventory below.
SOURCES = {
    "inspections": "food_inspections",
    "licenses": "food_licenses",
    "requests_311": "requests_311",
}

# A visit within this many days after a failed inspection is treated as a follow-up, not routine
FOLLOWUP_WINDOW_DAYS = 45

# Time-based split: last TEST_MONTHS = test, the VALID_MONTHS before that = validation
TEST_MONTHS = 12
VALID_MONTHS = 12

# 311 features (spatial counts). Turn off if 311 isn't loaded yet.
BUILD_311_FEATURES = True
RADIUS_M = 250
WINDOW_311_DAYS = 90

## 1. What's in the bucket

In [ ]:
if LOCAL_DATA_DIR:
    import fsspec

    fs = fsspec.filesystem("file")
    ROOT = os.path.abspath(os.path.join(LOCAL_DATA_DIR))
else:
    import gcsfs

    fs = gcsfs.GCSFileSystem(project=PROJECT_ID)
    ROOT = f"{BUCKET}/{RAW_PREFIX}"

print(f"Reading from: {ROOT}\n")
for src in sorted(fs.ls(ROOT)):
    name = src.rstrip("/").split("/")[-1]
    parts = sorted(p.rstrip("/").split("/")[-1] for p in fs.ls(src) if "load_date=" in p)
    n_files = len(fs.find(src))
    latest = parts[-1] if parts else "(no load_date partitions)"
    print(f"{name:<28} files={n_files:<5} partitions={len(parts):<4} latest={latest}")

In [ ]:
DATA_EXT = (".json", ".jsonl", ".ndjson", ".csv", ".parquet")


def latest_partition(source):
    base = f"{ROOT}/{source}"
    parts = [p for p in fs.ls(base) if "load_date=" in p.rstrip("/").split("/")[-1]]
    return sorted(parts)[-1] if parts else base


def read_file(path):
    with fs.open(path) as fh:
        if path.endswith(".csv"):
            return pd.read_csv(fh, low_memory=False)
        if path.endswith(".parquet"):
            return pd.read_parquet(fh)
        try:
            return pd.read_json(fh, lines=True, dtype=False)
        except ValueError:
            pass
    with fs.open(path) as fh:  # fall back to a regular JSON array
        return pd.read_json(fh, dtype=False)


def read_source(key):
    """Load the latest snapshot of a source. Returns None if it isn't in the bucket yet."""
    source = SOURCES[key]
    if not fs.exists(f"{ROOT}/{source}"):
        print(f"[{key}] not found at {ROOT}/{source}, skipping")
        return None
    path = latest_partition(source)
    files = [f for f in fs.find(path) if f.endswith(DATA_EXT)]
    df = pd.concat([read_file(f) for f in files], ignore_index=True)
    df.columns = [str(c).strip().lower() for c in df.columns]
    print(f"[{key}] {len(df):,} rows x {df.shape[1]} cols from {path} ({len(files)} file(s))")
    return df


insp_raw = read_source("inspections")
lic_raw = read_source("licenses")
r311_raw = read_source("requests_311")

### Column helper

Field names on data.boston.gov aren't documented consistently, so `pick()` takes the first name that exists
and fails loudly with the real column list if none do. If you hit a `KeyError`, add the real name to the list.

In [ ]:
def pick(df, *candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of {candidates} found. Columns are: {list(df.columns)}")
    return None


def to_naive_datetime(s):
    out = pd.to_datetime(s, errors="coerce", utc=True)
    return out.dt.tz_convert(None)


def parse_lat_lon(df):
    """Return (lat, lon) Series from lat/long columns or a '(lat, lon)' location string."""
    lat_c = pick(df, "latitude", "lat", "y", required=False)
    lon_c = pick(df, "longitude", "long", "lon", "lng", "x", required=False)
    if lat_c and lon_c:
        return pd.to_numeric(df[lat_c], errors="coerce"), pd.to_numeric(df[lon_c], errors="coerce")
    loc_c = pick(df, "location", "geocoded_location", required=False)
    if loc_c:
        nums = df[loc_c].astype(str).str.extract(r"(-?\d+\.\d+)[,\s]+(-?\d+\.\d+)").astype(float)
        return nums[0], nums[1]
    return pd.Series(np.nan, index=df.index), pd.Series(np.nan, index=df.index)


def in_boston(lat, lon):
    return lat.between(42.2, 42.42) & lon.between(-71.2, -70.95)

In [ ]:
insp_raw.info(verbose=True, show_counts=True)
insp_raw.head(3).T

## 2. Recency check (biggest project risk)

In [ ]:
C = {
    "license": pick(insp_raw, "licenseno", "license_no", "license_number", "licno"),
    "business": pick(insp_raw, "businessname", "business_name", "dbaname", required=False),
    "date": pick(insp_raw, "resultdttm", "result_dttm", "resultdate", "inspection_date", "inspdate"),
    "result": pick(insp_raw, "result", "result_code", "inspection_result"),
    "level": pick(insp_raw, "viollevel", "viol_level", "violation_level", "level"),
    "viol_code": pick(insp_raw, "violation", "violation_code", "viol_code", required=False),
    "viol_desc": pick(insp_raw, "violdesc", "violation_description", "viol_desc", required=False),
    "viol_status": pick(insp_raw, "violstatus", "viol_status", "violation_status", required=False),
    "comments": pick(insp_raw, "comments", "comment", required=False),
    "licensecat": pick(insp_raw, "licensecat", "license_cat", "license_category", required=False),
    "descript": pick(insp_raw, "descript", "description", "license_description", required=False),
}
C

In [ ]:
insp = insp_raw.copy()
insp["insp_dt"] = to_naive_datetime(insp[C["date"]])

latest = insp["insp_dt"].max()
age_days = (pd.Timestamp.now() - latest).days
print(f"Date range: {insp['insp_dt'].min():%Y-%m-%d} to {latest:%Y-%m-%d}")
print(f"Latest record is {age_days} days old")
print(f"Unparseable dates: {insp['insp_dt'].isna().sum():,}")
if age_days > 60:
    print("\n*** WARNING: no inspections in the last 60 days. Tell the team before building on this. ***")

monthly = insp.dropna(subset=["insp_dt"]).set_index("insp_dt").resample("MS").size()
ax = monthly[monthly.index >= latest - pd.DateOffset(years=5)].plot(figsize=(11, 3), title="Violation rows per month (last 5 years)")
ax.set_xlabel("")
plt.show()

insp["insp_dt"].dt.year.value_counts().sort_index().to_frame("rows")

**Record the latest date and row counts in `docs/data_sources.md`.**

## 3. Column profiles

In [ ]:
for key in ["result", "level", "licensecat", "descript", "viol_status"]:
    col = C.get(key)
    if col:
        print(f"--- {key} ({col}) ---")
        print(insp[col].value_counts(dropna=False).head(20).to_string(), "\n")

if C["licensecat"] and C["descript"]:
    display(pd.crosstab(insp[C["licensecat"]], insp[C["descript"]]))

In [ ]:
null_rates = insp_raw.isna().mean().sort_values(ascending=False).to_frame("null_rate")
null_rates[null_rates["null_rate"] > 0]

## 4. Reshape: violation rows → one row per inspection

Grain = `license` + inspection **date**. Each row of the raw data is one violation, so an inspection with
five violations shows up five times, and a clean inspection usually shows up once with an empty violation.

In [ ]:
v = insp.dropna(subset=["insp_dt"]).copy()
v["license"] = v[C["license"]].astype(str).str.strip()
v = v[v["license"].ne("") & v["license"].str.lower().ne("nan")]
v["insp_date"] = v["insp_dt"].dt.normalize()

level_str = v[C["level"]].astype(str).str.strip()
v["is_critical"] = level_str.str.count(r"\*") >= 3
v["has_violation"] = level_str.str.contains(r"\*")
v["result_fail"] = v[C["result"]].astype(str).str.contains("fail", case=False)
v["lat"], v["lon"] = parse_lat_lon(v)

agg = {
    "n_rows": ("license", "size"),
    "n_violations": ("has_violation", "sum"),
    "n_critical": ("is_critical", "sum"),
    "result_fail": ("result_fail", "max"),
    "result": (C["result"], "first"),
    "lat": ("lat", "median"),
    "lon": ("lon", "median"),
}
if C["business"]:
    agg["business"] = (C["business"], "first")
if C["licensecat"]:
    agg["licensecat"] = (C["licensecat"], "first")

inspections = (
    v.groupby(["license", "insp_date"], as_index=False)
    .agg(**agg)
    .sort_values(["license", "insp_date"])
    .reset_index(drop=True)
)
print(f"{len(v):,} violation rows -> {len(inspections):,} inspections across {inspections['license'].nunique():,} establishments")
inspections.head()

## 5. Target definition

- **`target`** = 1 if the inspection recorded any critical (`***`) violation, else 0
- **`result_fail`** = the city's own result code contains "fail" (kept to compare against)
- **Follow-ups** = a visit within `FOLLOWUP_WINDOW_DAYS` after a failed one. They're easy to predict and would
  inflate the metrics, so they're excluded from training/eval but still count as history for features.

The dataset doesn't label inspection type, so the follow-up rule is a heuristic. Check the distribution of
`days_since_prev` below and adjust the window if there's a clear bump.

In [ ]:
g = inspections.groupby("license")
inspections["target"] = (inspections["n_critical"] > 0).astype(int)
inspections["prev_date"] = g["insp_date"].shift(1)
inspections["days_since_prev"] = (inspections["insp_date"] - inspections["prev_date"]).dt.days
inspections["prev_result_fail"] = g["result_fail"].shift(1)
inspections["prev_target"] = g["target"].shift(1)

prev_failed = inspections["prev_result_fail"].eq(True) | inspections["prev_target"].eq(1)
inspections["is_followup"] = prev_failed & inspections["days_since_prev"].le(FOLLOWUP_WINDOW_DAYS)
inspections["is_routine"] = ~inspections["is_followup"]

print(inspections["is_followup"].value_counts(normalize=True).rename("share").to_string(), "\n")
print("Target vs result code (routine inspections only):")
routine = inspections[inspections["is_routine"]]
display(pd.crosstab(routine["target"], routine["result_fail"], margins=True))

fig, ax = plt.subplots(figsize=(11, 3))
inspections.loc[prev_failed, "days_since_prev"].clip(upper=180).hist(bins=90, ax=ax)
ax.axvline(FOLLOWUP_WINDOW_DAYS, color="red", ls="--")
ax.set_title("Days from a failed inspection to the next visit (red = follow-up cutoff)")
plt.show()

In [ ]:
routine = inspections[inspections["is_routine"]].copy()
routine["year"] = routine["insp_date"].dt.year
print(f"Base rate (routine inspections with a critical violation): {routine['target'].mean():.1%}\n")

by_year = routine.groupby("year")["target"].agg(["size", "mean"]).rename(columns={"size": "inspections", "mean": "critical_rate"})
display(by_year)
if "licensecat" in routine:
    display(routine.groupby("licensecat")["target"].agg(["size", "mean"]).sort_values("size", ascending=False))

## 6. Leakage-safe features

Every feature uses only inspections **before** the current one (`shift(1)` or `closed="left"` windows).
History is computed on *all* inspections, including follow-ups, then we keep the routine rows.

In [ ]:
f = inspections.copy()
g = f.groupby("license")

f["n_prior_insp"] = g.cumcount()
f["prior_critical_total"] = g["n_critical"].cumsum() - f["n_critical"]
f["prior_violations_total"] = g["n_violations"].cumsum() - f["n_violations"]
f["prior_critical_rate"] = (g["target"].cumsum() - f["target"]) / f["n_prior_insp"].replace(0, np.nan)
f["prev_n_critical"] = g["n_critical"].shift(1)
f["prev_n_violations"] = g["n_violations"].shift(1)
f["prev_result_fail"] = f["prev_result_fail"].astype(float)
f["days_since_first_insp"] = (f["insp_date"] - g["insp_date"].transform("min")).dt.days
f["month"] = f["insp_date"].dt.month

# The visit right before a routine inspection is often a follow-up that passed, which hides the
# original failure. So also look back to the last *routine* inspection.
routine_crit = f["n_critical"].where(f["is_routine"])
routine_date = f["insp_date"].where(f["is_routine"])
f["last_routine_n_critical"] = routine_crit.groupby(f["license"]).shift(1).groupby(f["license"]).ffill()
last_routine_date = routine_date.groupby(f["license"]).shift(1).groupby(f["license"]).ffill()
f["days_since_last_routine"] = (f["insp_date"] - last_routine_date).dt.days
f["had_followup_since_last_routine"] = f["prev_date"].gt(last_routine_date).astype(float).where(last_routine_date.notna())


def rolling_prior(df, col, window):
    r = (
        df.set_index("insp_date")
        .groupby("license")[col]
        .rolling(window, closed="left")
        .sum()
        .rename(f"{col}_{window}")
        .reset_index()
    )
    return df.merge(r, on=["license", "insp_date"], how="left")


for col in ["n_critical", "n_violations"]:
    f = rolling_prior(f, col, "365D")
f = f.rename(columns={"n_critical_365D": "critical_past_365d", "n_violations_365D": "violations_past_365d"})

# Sanity check: features must not peek at the current inspection
assert (f.loc[f["n_prior_insp"] == 0, "prior_critical_total"] == 0).all()
assert f["critical_past_365d"].fillna(0).le(f["prior_critical_total"]).all()
f.filter(like="prior").describe().T

## 7. Licenses and 311 checks

In [ ]:
lic = None
if lic_raw is not None:
    L = {
        "license": pick(lic_raw, "licenseno", "license_no", "license_number", "licno"),
        "status": pick(lic_raw, "licstatus", "license_status", "status", required=False),
        "issued": pick(lic_raw, "license_add_dt_tm", "issdttm", "issue_date", "licenseadddttm", required=False),
        "cat": pick(lic_raw, "licensecat", "license_cat", required=False),
    }
    print(L)
    lic = lic_raw.copy()
    lic["license"] = lic[L["license"]].astype(str).str.strip()
    if L["status"]:
        print(lic[L["status"]].value_counts(dropna=False).to_string())

    active = set(lic["license"])
    inspected = set(inspections["license"])
    print(f"\nActive licenses: {len(active):,}")
    print(f"Active licenses with at least one inspection: {len(active & inspected) / len(active):.1%}")
    print(f"Inspected establishments still active: {len(active & inspected) / len(inspected):.1%}")
    print("(A low first number usually means the license number formats differ between the two feeds.)")

    if L["issued"]:
        lic["license_issued"] = to_naive_datetime(lic[L["issued"]])
        f = f.merge(lic[["license", "license_issued"]].drop_duplicates("license"), on="license", how="left")
        f["license_age_days"] = (f["insp_date"] - f["license_issued"]).dt.days
        f.loc[f["license_age_days"] < 0, "license_age_days"] = np.nan

In [ ]:
r311 = None
FOOD_311 = r"rodent|rat\b|pest|trash|garbage|litter|dumpster|sanit|unsanitary|food|restaurant|odor"

if r311_raw is not None:
    R = {
        "type": pick(r311_raw, "type", "case_type", "request_type", "service_name"),
        "reason": pick(r311_raw, "reason", "category", "subject", required=False),
        "opened": pick(r311_raw, "open_dt", "opened", "open_date", "requested_datetime", "created_at"),
    }
    print(R)
    r311 = r311_raw.copy()
    r311["opened"] = to_naive_datetime(r311[R["opened"]])
    r311["lat"], r311["lon"] = parse_lat_lon(r311)
    print(f"Date range: {r311['opened'].min():%Y-%m-%d} to {r311['opened'].max():%Y-%m-%d}")

    types = r311[R["type"]].value_counts()
    print("\nFood-safety-related types (edit FOOD_311 to tune):")
    print(types[types.index.str.contains(FOOD_311, case=False, regex=True)].head(30).to_string())

    # Geocoding sanity: a single very common point is usually a default city-hall geocode
    good = in_boston(r311["lat"], r311["lon"])
    top_pt = r311.loc[good].groupby(["lat", "lon"]).size().sort_values(ascending=False).head(3)
    print(f"\nRows with usable Boston coordinates: {good.mean():.1%}")
    print("Most common coordinates (watch for a default point):")
    print(top_pt.to_string())

### 311 feature: food-safety complaints near each establishment

Counts complaints within `RADIUS_M` meters in the `WINDOW_311_DAYS` days **before** each inspection.
This is the pandas prototype; the production version should live in dbt (`ST_DWITHIN` in BigQuery).

In [ ]:
f["complaints_311_near"] = np.nan

if BUILD_311_FEATURES and r311 is not None:
    from sklearn.neighbors import BallTree

    EARTH_R = 6_371_000
    c = r311[
        r311[R["type"]].astype(str).str.contains(FOOD_311, case=False, regex=True)
        & in_boston(r311["lat"], r311["lon"])
        & r311["opened"].notna()
    ]
    # drop the default geocode point if one point holds >1% of complaints
    pt_counts = c.groupby(["lat", "lon"])["opened"].transform("size")
    c = c[pt_counts < max(50, 0.01 * len(c))]
    print(f"Using {len(c):,} food-safety 311 complaints")

    c_dates = c["opened"].values.astype("datetime64[D]")
    tree = BallTree(np.radians(c[["lat", "lon"]].values), metric="haversine")

    ok = in_boston(f["lat"], f["lon"])
    q = f.loc[ok, ["lat", "lon", "insp_date"]]
    counts = np.full(len(q), np.nan)
    window = np.timedelta64(WINDOW_311_DAYS, "D")
    for start in range(0, len(q), 5000):
        chunk = q.iloc[start : start + 5000]
        neighbors = tree.query_radius(np.radians(chunk[["lat", "lon"]].values), r=RADIUS_M / EARTH_R)
        dates = chunk["insp_date"].values.astype("datetime64[D]")
        for i, idx in enumerate(neighbors):
            d = c_dates[idx]
            counts[start + i] = np.count_nonzero((d < dates[i]) & (d >= dates[i] - window))
    f.loc[ok, "complaints_311_near"] = counts
    print(f["complaints_311_near"].describe().to_string())
else:
    print("Skipping 311 features")

## 8. Time-based split and baselines

Train on older inspections, validate on the next block, test on the most recent `TEST_MONTHS`.
Metrics that matter for a ranking problem:

- **ROC AUC / PR AUC**: overall ranking quality
- **Precision @ top 20%**: of the establishments we'd send inspectors to first, how many actually had a critical violation?
- **Lift @ top 20%**: that precision divided by the base rate

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

NUMERIC = [
    "n_prior_insp", "days_since_prev", "days_since_first_insp", "prev_n_critical", "prev_n_violations",
    "prev_result_fail", "last_routine_n_critical", "days_since_last_routine", "had_followup_since_last_routine",
    "prior_critical_total", "prior_violations_total", "prior_critical_rate",
    "critical_past_365d", "violations_past_365d", "month", "license_age_days", "complaints_311_near",
]
CATEGORICAL = ["licensecat"]
NUMERIC = [c for c in NUMERIC if c in f and f[c].notna().any()]
CATEGORICAL = [c for c in CATEGORICAL if c in f]

model_df = f[f["is_routine"]].copy()
end = model_df["insp_date"].max()
test_start = end - pd.DateOffset(months=TEST_MONTHS)
valid_start = test_start - pd.DateOffset(months=VALID_MONTHS)

train = model_df[model_df["insp_date"] < valid_start]
valid = model_df[(model_df["insp_date"] >= valid_start) & (model_df["insp_date"] < test_start)]
test = model_df[model_df["insp_date"] >= test_start]

split_summary = pd.DataFrame({
    s: {"from": d["insp_date"].min().date(), "to": d["insp_date"].max().date(), "rows": len(d), "base_rate": round(d["target"].mean(), 3)}
    for s, d in [("train", train), ("valid", valid), ("test", test)]
}).T
print("Features:", NUMERIC + CATEGORICAL)
split_summary

In [ ]:
def preprocessor(scale):
    num_steps = [("impute", SimpleImputer(strategy="median", add_indicator=True))]
    if scale:
        num_steps.append(("scale", StandardScaler()))
    transformers = [("num", Pipeline(num_steps), NUMERIC)]
    if CATEGORICAL:
        cat = Pipeline([
            ("impute", SimpleImputer(strategy="constant", fill_value="missing")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=20)),
        ])
        transformers.append(("cat", cat, CATEGORICAL))
    return ColumnTransformer(transformers, sparse_threshold=0)


def x(df):
    out = df[NUMERIC + CATEGORICAL].copy()
    for c in CATEGORICAL:
        out[c] = out[c].astype(object).where(out[c].notna(), "missing").astype(str)
    return out


def evaluate(name, y, score, top=0.20):
    k = max(1, int(len(y) * top))
    top_idx = np.argsort(-np.asarray(score))[:k]
    p_at_k = np.asarray(y)[top_idx].mean()
    return {
        "model": name,
        "roc_auc": roc_auc_score(y, score),
        "pr_auc": average_precision_score(y, score),
        f"precision@{int(top * 100)}%": p_at_k,
        f"lift@{int(top * 100)}%": p_at_k / np.mean(y),
    }


models = {
    "logistic_regression": Pipeline([("prep", preprocessor(scale=True)), ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))]),
    "hist_gradient_boosting": Pipeline([("prep", preprocessor(scale=False)), ("clf", HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, early_stopping=True, random_state=882))]),
}

results = []
for split_name, split in [("valid", valid), ("test", test)]:
    # Rule-of-thumb baselines: rank by past critical violations
    results.append({"split": split_name, **evaluate("rule: last_routine_n_critical", split["target"], split["last_routine_n_critical"].fillna(0))})
    results.append({"split": split_name, **evaluate("rule: critical_past_365d", split["target"], split["critical_past_365d"].fillna(0))})

for name, pipe in models.items():
    pipe.fit(x(train), train["target"])
    for split_name, split in [("valid", valid), ("test", test)]:
        results.append({"split": split_name, **evaluate(name, split["target"], pipe.predict_proba(x(split))[:, 1])})

pd.DataFrame(results).set_index(["split", "model"]).round(3)

**How to read this:** the models need to clearly beat the rule-of-thumb rows, or the extra features aren't
pulling their weight. Report test numbers only once you're done tuning on validation.

In [ ]:
from sklearn.inspection import permutation_importance

best = models["hist_gradient_boosting"]
imp = permutation_importance(best, x(valid), valid["target"], scoring="average_precision", n_repeats=5, random_state=882)
importance = pd.Series(imp.importances_mean, index=NUMERIC + CATEGORICAL).sort_values()
ax = importance.plot.barh(figsize=(8, 5), title="Permutation importance (validation, PR AUC drop)")
plt.show()

## 9. Save the feature table and next steps

Saves to `data/` (gitignored). The dbt feature mart should eventually reproduce this table.

**Next steps**
- [ ] Record the latest inspection date and real column names in `docs/data_sources.md`
- [ ] Hand Kai the confirmed column names for `stg_food_inspections.sql`
- [ ] Port the inspection grain + features into dbt (`int_inspections`, `fct_inspection_features`)
- [ ] Notebook 02: Chicago-style evaluation (how many days sooner are critical violations found?)
- [ ] MLflow tracking and hyperparameter tuning

In [ ]:
OUT_DIR = os.getenv("FEATURE_OUT_DIR", "../../data")
os.makedirs(OUT_DIR, exist_ok=True)
out_path = os.path.join(OUT_DIR, "features_v0.parquet")
keep = ["license", "insp_date", "target", "result_fail", "is_routine", "lat", "lon"] + NUMERIC + CATEGORICAL
f[keep].to_parquet(out_path, index=False)
print(f"Saved {len(f):,} rows to {out_path}")